# 02 — Does test look like train?

Workflow step 3. Everything after this depends on one premise:

> a held-out fold of training rows is a fair stand-in for the test set

If that is false, the CV score measures performance on a population we are not scored on,
and six weeks of tuning chase a number that does not track the leaderboard. This notebook
tests the premise three ways, from weakest evidence to strongest.

| | Question | Can see |
|---|---|---|
| 1 | Does any **column** differ? | marginals only |
| 2 | Do nulls **clump on rows**? | one specific joint property |
| 3 | Does **anything** differ? | the full joint distribution |

One quantity is used throughout. $N_{tr}$ and $N_{te}$ are the row counts, and

$$\pi \;=\; \frac{N_{te}}{N_{tr} + N_{te}} \;=\; \frac{295{,}753}{985{,}841} \;=\; 0.300$$

is the share of all rows that came from test. Under no shift, **every** subset of rows —
every bin, every category, every leaf of a tree — should be about $\pi$ test. Most of what
follows is one question asked of different subsets: *is this group 30% test?*

All logic lives in `src/s6e7/`; this notebook only calls and displays.

In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl

from s6e7 import adversarial, eda, io

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_float_precision(6)

train, test = io.load_train(), io.load_test()

# None = full data (the real answer). Set to 0.1 for a ~30 s pass while iterating;
# a subsample can only ever WEAKEN evidence of a shift, never manufacture it.
SAMPLE = None

print(f"train {train.height:,} rows   test {test.height:,} rows")
print(f"pi (global test share) = {test.height / (train.height + test.height):.6f}")

---
## 0. Why `id` is excluded from everything below

Competition ids are handed out per file, so they occupy disjoint contiguous ranges. A
single split on `id` separates train from test perfectly — AUC 1.0 that says nothing
whatever about the features. Confirm it once, then never feed it to the classifier.

In [ ]:
a, b = train["id"], test["id"]
print(f"train id  {a.min():>7,} – {a.max():>7,}   ({a.n_unique():,} unique)")
print(f"test  id  {b.min():>7,} – {b.max():>7,}   ({b.n_unique():,} unique)")
print(f"overlap   {len(set(a.to_list()) & set(b.to_list()))}")

---## 1. Do any columns differ? — the marginal checkThe check most people mean by "plot train vs test". Five numbers per column, all from`eda.numeric_shift`. Each exists to catch a failure the others are blind to.**`null_gap` — do both files lose this column equally often?**```pythonnull_gap = 100 * test[col].null_count() / test.height \         - 100 * train[col].null_count() / train.height````bmi`: train 13,898 / 690,088 = 2.01395%, test 5,956 / 295,753 = 2.01384%, gap**−0.00010 pp**. Two files of very different size agreeing to five decimals is not "twosimilar datasets" — it is one generator run twice with the same null probability. All 13columns look like this, which is exactly what makes section 2 so strange.**`gap_sd` — are the two centres in the same place?**```pythongap_sd = (test[col].mean() - train[col].mean()) / train[col].std()```Dividing by the column's own spread is the whole point: it turns "0.6 steps" and "0.01hours" into one comparable unit — *typical deviations apart*. Largest here: 0.0057.**`sd_ratio` — are they equally wide?** `test[col].std() / train[col].std()`, where 1.0 isidentical. It exists because two distributions can share a mean and differ completely inwidth, a case `gap_sd` scores as exactly zero. All seven land within 0.8% of 1.0.**`max_bin_dev` — does any slice of the range hold too many test rows?**The only one of the five that can see a change in *shape*:1. Pool both files' values for the column and sort them.2. Cut into 50 slices holding equal numbers of rows — lowest 2%, next 2%, …3. In each slice, what fraction of the rows came from test?4. Every slice should sit at π = 0.300. Report the worst.`heart_rate`, the worst of the seven:    worst slice = lowest 2% of heart rates, 50.0 – 58.6 bpm    19,421 rows:  13,900 train  +  5,521 test    expected test 5,826   ->   actual 5,521   ->   305 short    share 0.2843  vs  0.3000   ->   max_bin_dev = 0.0157Centre and width match train's exactly, and the bottom slice is *still* short. That is theshape shift the first three numbers cannot report — and at 1.6 points against a 30-pointbase, also far too small to act on.**`p_value` — could that be luck?** Same 50 slices, but instead of the single worst one,add up every slice's shortfall into one total and ask how often chance alone would producea total that big. `sleep_duration` 0.9405 (ordinary), `heart_rate` 0.0002 (2 in 10,000).> **Read `max_bin_dev`, not `p_value`.** For `heart_rate` the p-value says the deviation is> *real*; `max_bin_dev` says it is *1.6 percentage points*. Both true — they answer> different questions. At N = 986,000 there is enough statistical power to certify effects> far too small to change any decision. **A p-value answers "is it real?", never "is it> big?"**

In [ ]:
eda.numeric_shift(train, test, io.NUMERIC_COLS).select(
    "column", "gap_sd", "sd_ratio", "null_gap", "p_value", "max_bin_dev"
)

For categoricals there is nothing to bin — **the levels already are the bins.** `diff` iseach level's share of its own file in percentage points, with nulls counted as their ownlevel so every column's shares sum to 100.

In [ ]:
eda.category_shift(train, test, io.CATEGORICAL_COLS).head(10)

**Verdict on the marginals: they pass.** Mean gaps under 0.006 SD, spread ratios within
0.8%, null rates identical to five decimals. The only visible movement is `gender`
(female −3.3 pp, other +3.1 pp) and `physical_activity_level` (under 1 pp).

If we stopped here we would write "train and test are one distribution" and freeze random
folds. Section 3 shows that would be wrong.

---## 2. Do nulls clump on the same rows? — one joint propertyEverything in section 1 was about **columns**. This is about **rows**. For each row, counthow many of its 13 fields are missing; call that count $K$.Section 1 already pinned down the *average* of $K$ — both files lose each column at thesame rate, so both must average the same nulls per row. What it left completely open ishow those nulls are **arranged**. The same total can be sprinkled one-per-row across manyrows, or piled several-per-row onto few.If each column drops values independently of the others, we can compute exactly what $K$should look like. No simulation:```pythonpmf = np.array([1.0])          # before any column: "0 nulls", with certaintyfor p in rates:                # one column at a time    pmf = np.convolve(pmf, [1.0 - p, p])```Read it as bookkeeping. Start knowing a row has 0 nulls. Bring in a column: everypossibility you are holding splits in two — the column is *present* (weight $1-p$, countunchanged) or *missing* (weight $p$, count $+1$). Thirteen columns, thirteen steps, andyou have the exact probability of every value of $K$ — rather than enumerating all$2^{13} = 8{,}192$ null masks. That is the `independent_pct` column, built from **train'sown** rates.Then the part that does the work:$$\mathbb{E}[K] = \sum_j p_j \qquad\qquad \operatorname{Var}(K) = \sum_j p_j\,(1 - p_j)$$**The mean is blind to clumping. Only the variance sees it.** Move nulls off one row andonto another: the column rates never change, so $\mathbb{E}[K]$ never changes — but $K$spreads out. So the entire diagnostic is one ratio, observed variance over$\sum_j p_j(1-p_j)$: about 1 means nulls fall independently, above 1 means they clump.

In [ ]:
eda.missingness_dispersion(train, test, io.FEATURE_COLS)

In [ ]:
eda.null_count_profile(train, test, io.FEATURE_COLS)

**There it is.** Same mean number of nulls per row to six decimals (0.651360 vs 0.651361),
but test's **variance is 32% above** what independence predicts, while train sits right on
it (0.9925×).

Test's missingness is **clustered**: more perfectly-complete rows *and* more
heavily-gutted rows, fewer rows with exactly one gap. Identical columns, different rows —
which is precisely the thing no per-column check could ever have reported.

---## 3. Does anything at all differ? — adversarial validationSections 1 and 2 each tested something specific, chosen in advance. This tests*everything at once*, including whatever we did not think to look for.Throw the real target away. Label each row 0 if it came from `train.csv` and 1 if from`test.csv`, then fit $\hat q(x) \approx P(y = 1 \mid x)$ out-of-fold — exactly as we wouldfit any model. Score it with AUC, which reads directly as a probability:$$\text{AUC} \;=\; P\bigl(\hat q(x^{te}) > \hat q(x^{tr})\bigr)$$*Draw one test row and one train row at random; how often does the model rank them theright way round?* At 0.5 it cannot order them at all — **nothing** distinguishes the files,random folds are fair, proceed. Higher means a difference exists somewhere in the jointdistribution.Because AUC only ever compares one test row against one train row, the 70/30 imbalancecancels out and needs no correction.This turns "are these two 986,000-row, 13-dimensional distributions the same?" into"what's the AUC?" — a question we already have tools for.

In [14]:
result = adversarial.run(train, test, sample_frac=SAMPLE)
print(result.report())

adversarial validation: 690,088 train vs 295,753 test, 13 features
  OOF AUC   0.6518   (per-fold sd 0.0012)
  per fold  0.6498, 0.6530, 0.6519, 0.6512, 0.6532

SHIFT — AUC 0.6518 > 0.55. Read the importances; revisit fold design before freezing.

  feature                     gain   splits
  water_intake              32.34%   10,677
  calorie_expenditure       19.90%    9,713
  bmi                       12.78%   13,119
  physical_activity_level   10.86%    4,187
  smoking_alcohol            7.29%    3,586
  step_count                 5.23%   12,063
  diet_type                  3.24%    2,248
  gender                     2.61%    2,441
  exercise_duration          1.54%   10,026
  sleep_duration             1.51%   10,770
  heart_rate                 1.42%    9,946
  sleep_quality              1.06%    2,535
  stress_level               0.22%    1,689


### Do NOT read that importance table as a shift rankingGain totals the loss reduction across the splits a feature was chosen for, so it counts**opportunities** as much as signal. A continuous column with 12,000 distinct valuesoffers vastly more candidate thresholds than a 3-level categorical, and gain rewards thatwhatever the truth is.The honest per-feature number is that feature's adversarial AUC **on its own** — samedefinition as above, same scale as the joint result, one column in the model:

In [ ]:
solo = adversarial.solo_auc(train, test, sample_frac=SAMPLE)
solo.join(result.importance.select("feature", "gain_pct"), on="feature").sort(
    "solo_auc", descending=True
)

Every feature alone is at chance; together they are not — best solo **0.522** against ajoint **0.652**. **That gap is the part of the shift no single column carries**: the partonly a joint search can find, and the part no plot can show.Note how badly gain ranks it. `water_intake` takes 32% of the gain and is at chance on itsown; `gender` takes 2.6% and is the most-shifted column in the table. Here the gainranking is close to *anti-correlated* with the truth.

---## 4. The control — is the harness lying?A positive adversarial result is a claim about the data. This is what stops it being aclaim about our code. Permute the labels — destroying any real association while keepingthe class balance exactly — then re-run the whole pipeline:```pythony_shuffled = np.random.default_rng(seed).permutation(y)```The shuffled label is independent of the features by construction, so the only achievablescore is 0.5. Any bug that leaks the label — a misaligned concat, a row-order assumption,an index mismatch — survives permutation and shows up here as an AUC above chance.**Must come back ≈ 0.5000.** Run it every time the headline AUC is not already 0.5.

In [ ]:
control = adversarial.shuffled_control(train, test, sample_frac=SAMPLE)
print(f"shuffled-label AUC  {control:.4f}   (must be ~0.5000)")

---## Conclusion| Check | Result ||---|---|| Numeric marginals | pass — centres within 0.006 SD || Null rates per column | pass — identical to 5 decimals || Category levels | near-pass — `gender` moves 3.3 pp || Correlations | pass — largest difference 0.006 || Solo adversarial AUC | pass — every feature at chance || **Nulls per row** | **fail — variance 1.32× independence** || **Joint adversarial AUC** | **fail — 0.6518, control 0.5002** |Train and test are **not** draws from one distribution. Train draws nulls independentlyper column; test lets them co-occur.**Folds are not frozen.** The i.i.d. premise behind a plain stratified split is dead asstated, and the fold design has to answer this shift before step 4.The transferable lesson is in `LEARNING.md` — *Marginals cannot see a joint shift*. Youcan plot 13 marginals. You cannot plot a 13-dimensional joint.